# Project 1: AI-Powered Emotion Detection from Text
This Jupyter Notebook demonstrates the complete workflow for building an AI model that reads text sentences/paragraphs and predicts the underlying emotion expressed (e.g., **happy, sad, angry, surprised, or neutral**).

## Tech Stack:
*   **Python**
*   **NumPy, Pandas** (Data manipulation)
*   **Scikit-Learn** (Model building, feature extraction, evaluation)
*   **NLTK** (Text Preprocessing: tokenization, stopwords removal, stemming)

---

## Step 1: Library Imports and Setup

In [ ]:
import pandas as pd
import numpy as np
import os
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.pipeline import Pipeline
import joblib

print("Libraries imported successfully!")

## Step 2: Dataset Collection & Loading
We are using the **CrowdFlower Emotion Dataset** containing tweets labeled with emotions. We download and clean the dataset, mapping the raw emotions to 5 key classes:
*   **happy** (mapped from: happiness, fun, enthusiasm, relief, love)
*   **sad** (mapped from: sadness, worry, boredom, empty)
*   **angry** (mapped from: anger, hate)
*   **surprised** (mapped from: surprise)
*   **neutral** (mapped from: neutral)

In [ ]:
# Load the cleaned dataset
data_file = 'emotions_clean.csv'
if not os.path.exists(data_file):
    # Download if it doesn't exist yet
    print("Downloading and cleaning raw dataset...")
    import download_data
    download_data.download_and_clean_data()

df = pd.read_csv(data_file)
df = df.dropna()
print(f"Loaded dataset with {len(df)} samples.")
print("\nClass distribution:")
print(df['label'].value_counts())
df.head()

## Step 3: Text Preprocessing
To prepare text for machine learning models, we implement a preprocessing pipeline with the following steps:
1.  **Lowercasing**: Convert all text to lower case.
2.  **Removing Special Characters**: Strip out URLs, user handles, and non-alphabetic characters using regex.
3.  **Tokenization**: Split text into individual words using NLTK.
4.  **Stopwords Removal**: Filter out common words (e.g., "the", "is", "at") that don't carry emotion.
5.  **Stemming**: Reduce words to their base form (e.g., "feeling", "feels" -> "feel") using NLTK's `PorterStemmer`.

In [ ]:
stemmer = PorterStemmer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    # 1. Lowercase
    text = str(text).lower()
    # 2. Remove URLs, handles, and special characters
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    
    # 3. Tokenization
    tokens = word_tokenize(text)
    
    # 4. Stopwords removal
    filtered_tokens = [w for w in tokens if w not in stop_words]
    
    # 5. Stemming
    stemmed_tokens = [stemmer.stem(w) for w in filtered_tokens]
    
    return ' '.join(stemmed_tokens)

# Demo on a single sentence
sample = "I am feeling SO excited about this! Check out @user123. It's awesome!"
print("Original: ", sample)
print("Preprocessed: ", preprocess_text(sample))

Now, we apply the preprocessing to our entire dataset. This may take about 15-20 seconds.

In [ ]:
print("Preprocessing the dataset text...")
df['cleaned_text'] = df['text'].apply(preprocess_text)
print("Preprocessing completed successfully!")
df[['text', 'cleaned_text', 'label']].head()

## Step 4: Training & Testing Splits
We split our preprocessed text dataset into:
*   **Training set (80%)** to train the machine learning models.
*   **Testing set (20%)** to evaluate how well the models generalize to unseen sentences.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df['cleaned_text'], df['label'], test_size=0.2, random_state=42, stratify=df['label']
)
print(f"Training instances: {len(X_train)}")
print(f"Testing instances: {len(X_test)}")

## Step 5: Feature Extraction & Model Building
Computers do not understand text directly, so we must convert text to numbers. We will test two methods:
1.  **Bag-of-Words (BoW)**: Counts how many times each word appears in the text.
2.  **TF-IDF (Term Frequency-Inverse Document Frequency)**: Weights words by how informative they are across the dataset.

We will pair these features with two standard classification algorithms:
*   **Naive Bayes** (fast, probabilistic classifier)
*   **Logistic Regression** (robust linear classifier)

In [ ]:
vectorizers = {
    'Bag-of-Words': CountVectorizer(),
    'TF-IDF': TfidfVectorizer()
}

models = {
    'Naive Bayes': MultinomialNB(),
    'Logistic Regression': LogisticRegression(max_iter=1000)
}

best_accuracy = 0.0
best_pipeline = None
best_name = ""

for vec_name, vec in vectorizers.items():
    for model_name, model in models.items():
        # Bundle the vectorizer and model inside a Pipeline
        pipeline = Pipeline([
            ('vectorizer', vec),
            ('classifier', model)
        ])
        
        # Train model
        pipeline.fit(X_train, y_train)
        
        # Evaluate model
        preds = pipeline.predict(X_test)
        acc = accuracy_score(y_test, preds)
        f1 = f1_score(y_test, preds, average='weighted')
        
        print(f"{model_name} + {vec_name} -> Accuracy: {acc:.4f}, F1-Score: {f1:.4f}")
        
        if acc > best_accuracy:
            best_accuracy = acc
            best_pipeline = pipeline
            best_name = f"{model_name} with {vec_name}"

print(f"\nBest Model: {best_name} (Accuracy: {best_accuracy:.4f})")

## Step 6: Model Evaluation
Let's look at the detailed classification metrics and generate a confusion matrix for our best performing model.

In [ ]:
# Print final classification report
y_pred = best_pipeline.predict(X_test)
print("Classification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
labels = sorted(df['label'].unique())
cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(cm, index=[f"Actual: {l}" for l in labels], columns=[f"Predicted: {l}" for l in labels])
print(cm_df.to_string())

## Step 7: Saving the Model & Predicting on Custom Input
We serialize the best pipeline (which contains both the TF-IDF vectorizer and the trained classifier) to a pickle file using `joblib`. We also demonstrate how to load the file and make predictions on brand new user inputs.

In [ ]:
# Save the model
model_file = 'best_emotion_model.pkl'
joblib.dump(best_pipeline, model_file)
print(f"Model saved to {model_file}!")

# Load the model back
loaded_model = joblib.load(model_file)

# Interactive test function
def predict_emotion(sentence):
    cleaned = preprocess_text(sentence)
    prediction = loaded_model.predict([cleaned])[0]
    print(f"Input: '{sentence}' -> Predicted Emotion: {prediction.upper()}")

# Test cases
predict_emotion("I am so excited and happy to see you!")
predict_emotion("I can't believe this happened, it's so unexpected!")
predict_emotion("This makes me feel so angry and annoyed.")
predict_emotion("I am sitting here alone, feeling quite low today.")
predict_emotion("Let me check the scheduled time for the meeting.")